# 🧾 AI Invoice Processing Agent — AI Legends 2026

**Тэмцээн:** AI agent automation  
**Зорилго:** Invoice зураг/PDF-ээс мэдээлэл олборлох, баталгаажуулах, ангилах, шийдвэр гаргах AI agent

## Pipeline архитектур
1. **Extract** — Claude Vision API ашиглан invoice-оос structured data олборлох
2. **Validate** — Master DB-тай тулган 5 төрлийн зөрчил шалгах
3. **Classify** — 10 санхүүгийн ангилалд хуваарилах
4. **Decide** — AUTO_POST / HUMAN_APPROVAL / DENY шийдвэр гаргах
5. **Q&A** — Нийт үр дүн дээр асуулт-хариулт


In [ ]:
# Install dependencies
!pip install anthropic pymupdf pillow -q

In [ ]:
import anthropic
import base64
import json
import sqlite3
import os
import re
import fitz  # PyMuPDF
from datetime import datetime, timedelta
from pathlib import Path
from IPython.display import display, HTML, Image
import time

# ⚠️ Kaggle дээр API key-ээ Secrets-д нэмнэ, эсвэл энд шууд оруулна
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
client = anthropic.Anthropic()
print("✅ Anthropic client initialized")

## 1. Master Database ачаалах

In [ ]:
def load_master_db(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM Vendors")
    vendors = [dict(row) for row in cursor.fetchall()]
    
    cursor.execute("SELECT * FROM Items")
    items = [dict(row) for row in cursor.fetchall()]
    
    cursor.execute("SELECT * FROM InvoiceCategories")
    categories = [dict(row) for row in cursor.fetchall()]
    
    cursor.execute("""
        SELECT i.*, ic.Name as CategoryName,
               GROUP_CONCAT(il.ItemID || ':' || il.Qty || ':' || il.UnitPrice || ':' || il.Total, '|') as LineItems
        FROM Invoices i
        LEFT JOIN InvoiceCategories ic ON i.InvoiceCategoryID = ic.ID
        LEFT JOIN InvoiceLines il ON i.ID = il.InvoiceID
        GROUP BY i.ID
    """)
    historical_invoices = [dict(row) for row in cursor.fetchall()]
    conn.close()
    
    return {"vendors": vendors, "items": items, "categories": categories, "historical_invoices": historical_invoices}

# ⚠️ Kaggle дээр өгөгдлийн зам өөрчлөгдөж болно
DATA_DIR = "/kaggle/input/ai-legends-2026-ai-agents-automation/"  
DB_PATH = os.path.join(DATA_DIR, "master_invoices_database.db")

# Хэрэв local дээр ажиллуулж байвал:
if not os.path.exists(DB_PATH):
    DATA_DIR = "./"  # current directory
    DB_PATH = os.path.join(DATA_DIR, "master_invoices_database.db")

master_db = load_master_db(DB_PATH)
print(f"✅ Vendors: {len(master_db['vendors'])}")
print(f"✅ Items: {len(master_db['items'])}")
print(f"✅ Categories: {len(master_db['categories'])}")
print(f"✅ Historical invoices: {len(master_db['historical_invoices'])}")

# Show vendors
for v in master_db['vendors']:
    print(f"  {v['Name']} | {v['Bank']} | {v['Account']}")

## 2. Invoice файл уншигч (JPG/PNG/PDF → base64)

In [ ]:
def load_invoice_file(filepath):
    ext = Path(filepath).suffix.lower()
    
    if ext in ['.jpg', '.jpeg', '.png']:
        media_type = "image/jpeg" if ext in ['.jpg', '.jpeg'] else "image/png"
        with open(filepath, 'rb') as f:
            data = base64.standard_b64encode(f.read()).decode('utf-8')
        return data, media_type
    
    elif ext == '.pdf':
        doc = fitz.open(filepath)
        page = doc[0]
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        doc.close()
        data = base64.standard_b64encode(img_bytes).decode('utf-8')
        return data, "image/png"
    
    else:
        raise ValueError(f"Unsupported file type: {ext}")

print("✅ File loader ready")

## 3. Claude Vision ашиглан мэдээлэл олборлох

In [ ]:
EXTRACTION_PROMPT = """You are an expert invoice data extraction system. Extract ALL information from this Mongolian invoice image.

Return ONLY valid JSON with this exact structure (no markdown, no explanation):
{
    "invoice_number": "string or null",
    "vendor_name": "string or null",
    "bank_name": "string or null",
    "account_number": "string or null",
    "email": "string or null",
    "invoice_date": "YYYY-MM-DD or null",
    "due_date": "YYYY-MM-DD or null",
    "line_items": [
        {
            "description": "string",
            "quantity": number,
            "unit_price": number,
            "total": number
        }
    ],
    "grand_total": number or null
}

Important rules:
- Read ALL text carefully, including handwritten text
- Numbers should be integers (no commas)
- Dates must be YYYY-MM-DD format (convert from any format)
- Vendor names follow pattern "Демо Компани-N" (sometimes handwritten as "Demo Компани-N")
- Bank names: "Демо Банк 1" or "Демо Банк 2" (sometimes "Demo bank 1/2" or "Demo банк 1/2")
- Account numbers are 10-digit numbers
- For handwritten invoices, be extra careful with number recognition
- Return ONLY the JSON object
"""

def extract_invoice_data(filepath, retries=2):
    img_data, media_type = load_invoice_file(filepath)
    
    for attempt in range(retries + 1):
        try:
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=2000,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": img_data}},
                        {"type": "text", "text": EXTRACTION_PROMPT}
                    ]
                }]
            )
            text = response.content[0].text.strip()
            text = re.sub(r'^```json\s*', '', text)
            text = re.sub(r'\s*```$', '', text)
            return json.loads(text)
        except Exception as e:
            if attempt < retries:
                print(f"  ⚠️ Retry {attempt+1}: {e}")
                time.sleep(2)
            else:
                raise e

print("✅ Extraction function ready")

## 4. Баталгаажуулалт (5 төрлийн зөрчил шалгах)

| # | Зөрчлийн төрөл | Тайлбар |
|---|---|---|
| 1 | `AMOUNT_MISMATCH` | qty × unit_price ≠ total, эсвэл lines sum ≠ grand_total |
| 2 | `UNREGISTERED_VENDOR` | Vendor master DB-д бүртгэлгүй |
| 3 | `BANK_ACCOUNT_MISMATCH` | Банк/данс vendor бүртгэлтэй таарахгүй |
| 4 | `INVALID_DATE` | Огноо буруу формат, due < invoice, хэт хуучин |
| 5 | `DUPLICATE` | Өмнөх invoice-тай vendor+date+total давхцаж байгаа |


In [ ]:
def validate_invoice(extracted, master_db):
    issues = []
    
    # --- AMOUNT MISMATCH ---
    if extracted.get("line_items"):
        for i, item in enumerate(extracted["line_items"]):
            qty = item.get("quantity", 0) or 0
            unit_price = item.get("unit_price", 0) or 0
            total = item.get("total", 0) or 0
            expected = qty * unit_price
            if expected != 0 and total != 0 and expected != total:
                issues.append({"type": "AMOUNT_MISMATCH",
                    "detail": f"Line {i+1}: {qty} × {unit_price:,} = {expected:,}, but shows {total:,}"})
        
        line_sum = sum(item.get("total", 0) or 0 for item in extracted["line_items"])
        grand_total = extracted.get("grand_total", 0) or 0
        if line_sum != 0 and grand_total != 0 and line_sum != grand_total:
            issues.append({"type": "AMOUNT_MISMATCH",
                "detail": f"Sum of lines ({line_sum:,}) ≠ grand total ({grand_total:,})"})
    
    # --- UNREGISTERED VENDOR ---
    vendor_name = (extracted.get("vendor_name") or "").strip()
    vendor_match = None
    for v in master_db["vendors"]:
        db_name = v["Name"].strip()
        norm_v = re.sub(r'\s*[-–—]\s*', '-', vendor_name.lower())
        norm_db = re.sub(r'\s*[-–—]\s*', '-', db_name.lower())
        if norm_v == norm_db:
            vendor_match = v
            break
    
    if not vendor_match:
        issues.append({"type": "UNREGISTERED_VENDOR",
            "detail": f"Vendor '{vendor_name}' not found in master database"})
    
    # --- BANK ACCOUNT MISMATCH ---
    if vendor_match:
        inv_bank = (extracted.get("bank_name") or "").strip()
        inv_account = (extracted.get("account_number") or "").strip()
        db_bank = vendor_match["Bank"].strip()
        db_account = vendor_match["Account"].strip()
        
        def normalize_bank(name):
            name = name.lower().strip()
            name = re.sub(r'\s+', ' ', name)
            name = name.replace('demo', 'демо').replace('bank', 'банк')
            return name
        
        if inv_bank and normalize_bank(inv_bank) != normalize_bank(db_bank):
            issues.append({"type": "BANK_ACCOUNT_MISMATCH",
                "detail": f"Bank: invoice='{inv_bank}', DB='{db_bank}'"})
        
        if inv_account and inv_account != db_account:
            issues.append({"type": "BANK_ACCOUNT_MISMATCH",
                "detail": f"Account: invoice='{inv_account}', DB='{db_account}'"})
    
    # --- INVALID DATE ---
    def parse_date(s):
        if not s: return None
        s = s.strip().replace('/', '-')
        try: return datetime.strptime(s, '%Y-%m-%d')
        except: return None
    
    inv_date = parse_date(extracted.get("invoice_date"))
    due_date = parse_date(extracted.get("due_date"))
    
    if extracted.get("invoice_date") and not inv_date:
        issues.append({"type": "INVALID_DATE", "detail": f"Cannot parse: '{extracted['invoice_date']}'"})
    if extracted.get("due_date") and not due_date:
        issues.append({"type": "INVALID_DATE", "detail": f"Cannot parse: '{extracted['due_date']}'"})
    if inv_date and due_date and due_date < inv_date:
        issues.append({"type": "INVALID_DATE", "detail": f"Due date before invoice date"})
    
    now = datetime(2026, 4, 29)
    if inv_date and (inv_date.year < 2020 or inv_date > now + timedelta(days=365)):
        issues.append({"type": "INVALID_DATE", "detail": f"Date {inv_date.date()} seems unreasonable"})
    
    # --- DUPLICATE ---
    grand_total = extracted.get("grand_total", 0) or 0
    inv_date_str = (extracted.get("invoice_date") or "").replace("/", "-")
    
    for hist in master_db["historical_invoices"]:
        hist_vendor = (hist.get("VendorName") or "").strip().lower()
        hist_date = (hist.get("InvoiceDate") or "").strip()
        hist_total = hist.get("GrandTotal", 0) or 0
        
        if (vendor_name.lower() == hist_vendor and 
            grand_total != 0 and grand_total == hist_total and
            inv_date_str == hist_date):
            issues.append({"type": "DUPLICATE",
                "detail": f"Matches historical invoice ID={hist['ID']}"})
            break
    
    return issues, vendor_match

print("✅ Validation ready")

## 5. Санхүүгийн ангилал тогтоох

In [ ]:
def classify_invoice(extracted, master_db, vendor_match):
    categories = master_db["categories"]
    items_db = master_db["items"]
    
    descriptions = []
    if extracted.get("line_items"):
        descriptions = [item.get("description", "") for item in extracted["line_items"]]
    combined_desc = " ".join(descriptions).lower()
    
    # Strategy 1: Vendor-ийн түүхэн ангилал
    if vendor_match:
        vendor_name = vendor_match["Name"]
        cat_counts = {}
        for hist in master_db["historical_invoices"]:
            if hist.get("VendorName", "").strip() == vendor_name:
                cid = hist.get("InvoiceCategoryID")
                if cid: cat_counts[cid] = cat_counts.get(cid, 0) + 1
        if cat_counts:
            best_id = max(cat_counts, key=cat_counts.get)
            for cat in categories:
                if cat["ID"] == best_id:
                    return cat["Name"], cat["ID"]
    
    # Strategy 2: Keyword matching
    keyword_map = {
        1: ["түрээс", "оффис түрээс", "форклифт"],
        2: ["цахилгаан", "дулаан", "ус", "хог", "цэвэрлэгээ", "харуул", "хамгаалалт", "агааржуулалт"],
        3: ["сервер", "интернэт", "лиценз", "програм", "кибер", "домэйн", "ssl", "нөөцлөлт", "вэб", "бараа бүртгэл"],
        4: ["монитор", "принтер", "камер", "кабель", "тоног"],
        5: ["тээвэр", "шатахуун", "хүргэлт", "ачаа", "гааль", "gps", "логистик"],
        6: ["агуулах", "хадгалалт", "боодол"],
        7: ["засвар", "лифт", "тагт", "цонх", "сантехник", "дээвэр", "хаалга", "техник", "барилга", "зам талбай"],
        8: ["сургалт", "хөгжүүлэлт", "мэргэжил"],
        9: ["даатгал"],
        10: ["зөвшөөрөл", "тусгай зөвшөөрөл", "лиценз сунгалт"]
    }
    
    best_id, best_score = None, 0
    for cid, kws in keyword_map.items():
        score = sum(1 for kw in kws if kw in combined_desc)
        if score > best_score:
            best_score, best_id = score, cid
    
    if best_id:
        for cat in categories:
            if cat["ID"] == best_id:
                return cat["Name"], cat["ID"]
    
    # Strategy 3: Item DB matching
    for desc in descriptions:
        desc_lower = desc.lower().strip()
        for mi in items_db:
            if desc_lower in mi["ItemName"].lower() or mi["ItemName"].lower() in desc_lower:
                # Map item → category (based on item ID ranges)
                iid = mi["ID"]
                cat_map = {(1,10):3, (10,14):4, (14,16):7, (16,17):8, (17,18):10,
                          (18,19):1, (19,26):2, (26,35):7, (35,42):5, (42,46):6,
                          (46,47):7, (47,48):9, (48,49):1, (49,50):3, (50,51):10}
                for (lo,hi), cid in cat_map.items():
                    if lo <= iid < hi:
                        for cat in categories:
                            if cat["ID"] == cid:
                                return cat["Name"], cat["ID"]
    
    return "Бусад", None

print("✅ Classification ready")

## 6. Эцсийн шийдвэр гаргах

In [ ]:
def compute_confidence(extracted):
    score = 1.0
    for field in ["vendor_name", "invoice_date", "grand_total", "account_number", "bank_name"]:
        if not extracted.get(field):
            score -= 0.15
    if not extracted.get("line_items"):
        score -= 0.20
    return max(0.1, min(1.0, round(score, 2)))

def make_decision(extracted, issues, vendor_match, master_db):
    # DENY: critical issues
    deny_types = {"AMOUNT_MISMATCH", "INVALID_DATE", "BANK_ACCOUNT_MISMATCH", "DUPLICATE"}
    critical = [i for i in issues if i["type"] in deny_types]
    if critical:
        return "DENY", [f"{i['type']}: {i['detail']}" for i in critical]

    # DENY: unregistered vendor
    if any(i["type"] == "UNREGISTERED_VENDOR" for i in issues):
        return "DENY", ["UNREGISTERED_VENDOR: Vendor not in master database"]

    # HUMAN_APPROVAL: low confidence extraction
    confidence = compute_confidence(extracted)
    if confidence < 0.6:
        return "HUMAN_APPROVAL", [f"Low extraction confidence ({confidence:.2f}): key fields missing"]

    # AUTO_POST: registered, clean, has history
    if vendor_match and len(issues) == 0:
        vendor_name = vendor_match["Name"]
        has_history = any(h.get("VendorName","").strip() == vendor_name
                        for h in master_db["historical_invoices"])
        if has_history:
            return "AUTO_POST", [f"Registered vendor, clean history, confidence={confidence:.2f}"]
        else:
            return "HUMAN_APPROVAL", ["Registered vendor but no historical invoices"]

    return "HUMAN_APPROVAL", ["Requires manual review"]

print("✅ Decision engine ready")

## 7. Бүх invoice боловсруулах

In [ ]:
def process_invoice(filepath):
    filename = os.path.basename(filepath)

    try:
        extracted = extract_invoice_data(filepath)
    except Exception as e:
        print(f"  ⚠️ {filename}: extraction failed — {e}")
        return {
            "filename": filename, "extracted": {}, "invoice_number": None,
            "vendor_name": None, "bank_name": None, "account_number": None,
            "email": None, "invoice_date": None, "due_date": None,
            "line_items": [], "grand_total": None, "category": "Unknown",
            "category_id": None, "issues": [{"type": "EXTRACTION_ERROR", "detail": str(e)}],
            "decision": "HUMAN_APPROVAL", "decision_reasons": [f"Extraction failed: {e}"],
            "confidence": 0.0
        }

    issues, vendor_match = validate_invoice(extracted, master_db)
    cat_name, cat_id = classify_invoice(extracted, master_db, vendor_match)
    decision, reasons = make_decision(extracted, issues, vendor_match, master_db)
    confidence = compute_confidence(extracted)

    status_icon = {"AUTO_POST": "✅", "HUMAN_APPROVAL": "⚠️", "DENY": "❌"}
    print(f"  {status_icon.get(decision,'?')} {filename}: {decision} | {cat_name} | conf={confidence:.2f} | issues={len(issues)}")

    return {
        "filename": filename,
        "invoice_number": extracted.get("invoice_number"),
        "vendor_name": extracted.get("vendor_name"),
        "bank_name": extracted.get("bank_name"),
        "account_number": extracted.get("account_number"),
        "email": extracted.get("email"),
        "invoice_date": extracted.get("invoice_date"),
        "due_date": extracted.get("due_date"),
        "line_items": extracted.get("line_items", []),
        "grand_total": extracted.get("grand_total"),
        "category": cat_name,
        "category_id": cat_id,
        "issues": issues,
        "decision": decision,
        "decision_reasons": reasons,
        "confidence": confidence
    }

# Get all invoice files
invoice_files = sorted([
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.startswith('invoice_') and f.split('.')[-1] in ('jpg', 'jpeg', 'png', 'pdf')
])

print(f"📄 Found {len(invoice_files)} invoices to process\n")

results = []
for fp in invoice_files:
    result = process_invoice(fp)
    results.append(result)
    time.sleep(0.5)

print(f"\n✅ Done! Processed {len(results)} invoices")

## 8. Нийт үр дүнгийн хураангуй

In [ ]:
def generate_summary(results):
    total = len(results)
    auto = [r for r in results if r["decision"] == "AUTO_POST"]
    approval = [r for r in results if r["decision"] == "HUMAN_APPROVAL"]
    denied = [r for r in results if r["decision"] == "DENY"]
    
    all_issues = [i for r in results for i in r.get("issues",[])]
    issue_counts = {}
    for i in all_issues:
        issue_counts[i["type"]] = issue_counts.get(i["type"], 0) + 1
    
    cat_counts = {}
    for r in results:
        c = r.get("category","Unknown")
        cat_counts[c] = cat_counts.get(c, 0) + 1
    
    total_amt = sum(r.get("grand_total",0) or 0 for r in results)
    denied_amt = sum(r.get("grand_total",0) or 0 for r in denied)
    
    return {
        "total_invoices": total,
        "auto_post_count": len(auto),
        "human_approval_count": len(approval),
        "deny_count": len(denied),
        "issue_breakdown": issue_counts,
        "category_distribution": cat_counts,
        "total_amount": total_amt,
        "denied_amount": denied_amt,
        "duplicate_count": issue_counts.get("DUPLICATE", 0),
        "unregistered_vendor_count": issue_counts.get("UNREGISTERED_VENDOR", 0),
        "amount_mismatch_count": issue_counts.get("AMOUNT_MISMATCH", 0),
        "bank_mismatch_count": issue_counts.get("BANK_ACCOUNT_MISMATCH", 0),
        "invalid_date_count": issue_counts.get("INVALID_DATE", 0),
    }

summary = generate_summary(results)

print("=" * 60)
print("📊 PROCESSING SUMMARY")
print("=" * 60)
print(f"Total invoices:     {summary['total_invoices']}")
print(f"✅ AUTO_POST:       {summary['auto_post_count']}")
print(f"⚠️  HUMAN_APPROVAL: {summary['human_approval_count']}")
print(f"❌ DENY:            {summary['deny_count']}")
print(f"\n🔍 Issues found:")
for t, c in summary['issue_breakdown'].items():
    print(f"   {t}: {c}")
print(f"\n💰 Total amount:  {summary['total_amount']:>15,}₮")
print(f"💰 Denied amount: {summary['denied_amount']:>15,}₮")
print(f"\n📁 Category distribution:")
for cat, cnt in sorted(summary['category_distribution'].items(), key=lambda x: -x[1]):
    print(f"   {cat}: {cnt}")

## 9. Aggregate Q&A — Нийт үр дүн дээрх асуулт-хариулт

In [ ]:
def ask_question(question):
    context = f"""You are an AI invoice processing analyst. You processed {summary['total_invoices']} invoices.

Summary: {json.dumps(summary, ensure_ascii=False)}

Detailed results: {json.dumps(results, ensure_ascii=False)}

Answer precisely with specific numbers. Answer in the same language as the question."""
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role": "user", "content": context + "\n\nQuestion: " + question}]
    )
    return response.content[0].text

# Жишээ асуултууд
questions = [
    "Хэдэн нэхэмжлэл DENY болсон бэ? Шалтгааныг нь жагсаа.",
    "Хэд нь duplicate байсан бэ?",
    "Хэд нь бүртгэлгүй vendor-той байсан бэ?",
    "Хамгийн их нэхэмжлэл ирүүлсэн vendor аль вэ?",
    "AUTO_POST болсон нэхэмжлэлүүдийн нийт дүн хэд вэ?"
]

for q in questions:
    print(f"\n❓ {q}")
    print(f"💬 {ask_question(q)}")
    print("-" * 60)

## 10. Үр дүнг хадгалах

In [ ]:
# Save full results as JSON
output = {"results": results, "summary": summary}
with open("invoice_results.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

# Save summary table as CSV
import csv
with open("invoice_summary.csv", "w", encoding="utf-8", newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        "filename", "invoice_number", "vendor_name", "invoice_date", "due_date",
        "grand_total", "category", "decision", "issues_count", "issue_types"
    ])
    writer.writeheader()
    for r in results:
        writer.writerow({
            "filename": r["filename"],
            "invoice_number": r.get("invoice_number"),
            "vendor_name": r.get("vendor_name"),
            "invoice_date": r.get("invoice_date"),
            "due_date": r.get("due_date"),
            "grand_total": r.get("grand_total"),
            "category": r.get("category"),
            "decision": r.get("decision"),
            "issues_count": len(r.get("issues", [])),
            "issue_types": ", ".join(set(i["type"] for i in r.get("issues", [])))
        })

print("✅ Results saved to invoice_results.json and invoice_summary.csv")

## 11. Үр дүнгийн visualization

In [ ]:
# Decision distribution pie chart (text-based for notebook)
print("\n📊 Decision Distribution:")
print(f"  AUTO_POST:      {'█' * summary['auto_post_count']} ({summary['auto_post_count']})")
print(f"  HUMAN_APPROVAL: {'█' * summary['human_approval_count']} ({summary['human_approval_count']})")
print(f"  DENY:           {'█' * summary['deny_count']} ({summary['deny_count']})")

print("\n📊 Issue Types:")
for t, c in sorted(summary['issue_breakdown'].items(), key=lambda x: -x[1]):
    print(f"  {t:30s} {'█' * c} ({c})")

# Show denied invoices detail
print("\n❌ DENIED Invoices Detail:")
for r in results:
    if r["decision"] == "DENY":
        issue_types = [i["type"] for i in r.get("issues", [])]
        print(f"  {r['filename']}: {r.get('vendor_name','?')} | {r.get('grand_total',0):,}₮ | {issue_types}")